# Мини-семинар. PyTorch за час: ровно то, что понадобится в RL

Этот ноутбук для тех, кто с PyTorch не работал или подзабыл. Он не заменяет курс по deep learning: здесь только то, что нужно, чтобы с недели 5 писать DQN, policy gradient и actor-critic.

Если слова «слой», «градиент» и «функция потерь» вам ничего не говорят, начните с `dl_basics.ipynb` в этой же папке: там нейросети объясняются с нуля, а здесь предполагается, что вы уже знаете, что такое обучение градиентным спуском.

**План**

1. Тензоры: создание, формы, numpy ↔ torch, индексация и `gather`
2. Autograd: `requires_grad`, `backward`, `no_grad`, `detach`
3. Нейросеть как `nn.Module`
4. Цикл обучения: loss, оптимизатор, `zero_grad / backward / step`
5. Первый «агент на нейросети»: клонируем эвристику для CartPole (behavior cloning)
6. Распределения: `Categorical`, `sample`, `log_prob`
7. Чек-лист типовых ошибок
8. Упражнения для самопроверки

Установка: `pip install torch` (CPU-версии достаточно для всего курса до недели 5 включительно; инструкции для GPU на [pytorch.org](https://pytorch.org/get-started/locally/)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 1. Тензоры

`torch.Tensor` — это numpy-массив, который (а) умеет считать градиенты и (б) может жить на GPU. Почти весь numpy-синтаксис работает как есть.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x, x.shape, x.dtype, x.device, sep="\n")

print(torch.zeros(2, 3))
print(torch.randn(2, 3))          # N(0, 1)
print(torch.arange(6).reshape(2, 3))

# numpy <-> torch: память общая, копирования нет
a = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(a)
t_back = t.numpy()
print(type(t), type(t_back))

**Важно про dtype.** numpy по умолчанию создаёт `float64`, PyTorch-слои ждут `float32`. Наблюдения из Gymnasium приходят как `float32`-массивы numpy, но результаты ваших вычислений в numpy (например, средние) легко становятся `float64`. Приводите явно: `torch.as_tensor(obs, dtype=torch.float32)`.

In [ ]:
obs = np.array([0.01, -0.2, 0.03, 0.4])           # float64!
t = torch.as_tensor(obs, dtype=torch.float32)
print(obs.dtype, "->", t.dtype)

# Батч: первая размерность — номер примера. В RL батч = набор переходов из буфера.
batch = torch.randn(32, 4)         # 32 наблюдения по 4 признака
print(batch.shape, batch.mean(dim=0).shape, batch.sum(dim=1).shape)

# Broadcasting работает как в numpy
print((batch - batch.mean(dim=0)).shape)

### `gather`: достать Q(s, a) для выбранных действий

В DQN сеть выдаёт Q-значения для **всех** действий сразу: тензор формы `(batch, n_actions)`. Нам нужны значения только для тех действий, которые реально были сделаны. Для этого `gather`.

In [ ]:
q_values = torch.tensor([[1.0, 5.0, 3.0],
                         [2.0, 0.5, 9.0],
                         [7.0, 1.0, 1.0]])            # (batch=3, n_actions=3)
actions = torch.tensor([1, 2, 0])                    # какое действие сделали в каждом переходе

q_taken = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
print(q_taken)                                       # ожидаем [5., 9., 7.]

# Эквивалент через fancy indexing:
print(q_values[torch.arange(3), actions])

# Для TD-target понадобится максимум по действиям:
print(q_values.max(dim=1).values, q_values.argmax(dim=1))

## 2. Autograd

Тензор с `requires_grad=True` запоминает все операции над собой. `backward()` на скаляре считает градиенты по всем таким тензорам и складывает их в `.grad`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x            # dy/dx = 2x + 2 = 8 при x = 3
y.backward()
print("dy/dx =", x.grad)

# Градиенты НАКАПЛИВАЮТСЯ: второй backward прибавит ещё 8
y = x ** 2 + 2 * x
y.backward()
print("после второго backward:", x.grad, "<- поэтому в цикле обучения нужен zero_grad()")
x.grad.zero_()

### `torch.no_grad()` и `detach()`

* `with torch.no_grad():` — не строить граф вычислений. Используем, когда просто **применяем** сеть: выбираем действие в среде, считаем target. Экономит память и время.
* `tensor.detach()` — отрезать тензор от графа: градиент через него не потечёт.

**Зачем это в RL.** В DQN лосс выглядит так:

$$
L = \big( Q_\theta(s, a) - \underbrace{[\,r + \gamma \max_{a'} Q_{\theta}(s', a')\,]}_{\text{target}} \big)^2 .
$$

Target тоже посчитан сетью, но мы хотим считать его **константой** и двигать только $Q_\theta(s,a)$. Иначе градиент потечёт в обе части и обучение станет неустойчивым. Отсюда `target = (r + gamma * q_next.max(1).values).detach()` или вычисление target под `no_grad`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
pred = w * 3.0                       # "Q(s,a)"
target = (w * 5.0).detach()          # "r + γ max Q(s', a')" — константа
loss = (pred - target) ** 2
loss.backward()
print("grad с detach   :", w.grad)   # 2 * (6 - 10) * 3 = -24

w.grad.zero_()
pred = w * 3.0
target = w * 5.0                     # забыли detach
loss = (pred - target) ** 2
loss.backward()
print("grad без detach :", w.grad)   # 2 * (6 - 10) * (3 - 5) = 16 — другой знак!

## 3. Нейросеть как `nn.Module`

Сеть — класс с параметрами и методом `forward`. `nn.Linear(in, out)` — полносвязный слой $y = xW^\top + b$. В RL почти всегда хватает MLP из 2–3 слоёв с `ReLU` или `Tanh`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)

net = MLP(in_dim=4, hidden=64, out_dim=2)
print(net)
n_params = sum(p.numel() for p in net.parameters())
print("параметров:", n_params)

x = torch.randn(8, 4)                # батч из 8 наблюдений CartPole
out = net(x)                         # forward вызывается неявно
print(out.shape)                     # (8, 2): по числу на каждое действие

## 4. Цикл обучения

Четыре строки, которые вы напишете сотни раз:

```python
loss = criterion(net(x), y)
optimizer.zero_grad()     # обнулить старые градиенты
loss.backward()           # посчитать новые
optimizer.step()          # сделать шаг
```

Потренируемся на регрессии: приблизим $y = \sin(3x)$ на отрезке.

In [ ]:
x_train = torch.linspace(-2, 2, 256).unsqueeze(1)          # (256, 1)
y_train = torch.sin(3 * x_train) + 0.1 * torch.randn_like(x_train)

reg = MLP(in_dim=1, hidden=64, out_dim=1)
optimizer = torch.optim.Adam(reg.parameters(), lr=1e-2)

losses = []
for step in range(600):
    pred = reg(x_train)
    loss = F.mse_loss(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())              # .item() -> python float, без графа

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(losses); axes[0].set_yscale("log"); axes[0].set_title("MSE loss"); axes[0].set_xlabel("шаг")
with torch.no_grad():
    axes[1].plot(x_train, y_train, ".", alpha=0.4, label="данные")
    axes[1].plot(x_train, reg(x_train), lw=2, label="сеть")
axes[1].legend(); axes[1].set_title("y = sin(3x)")
plt.show()

## 5. Первый «агент на нейросети»: клонируем эвристику для CartPole

На лекции была эвристическая политика для CartPole: «толкай тележку в ту сторону, куда падает шест». Сделаем следующее:

1. Соберём датасет пар (наблюдение, действие эвристики), запуская эвристику в среде.
2. Обучим сеть предсказывать действие по наблюдению (обычная классификация на 2 класса).
3. Запустим сеть как политику и посмотрим, держит ли она шест.

Это **behavior cloning** — простейшая форма imitation learning (неделя 11). RL здесь ещё нет: сеть учится у «учителя», а не у награды. Но механика (наблюдение → сеть → действие) ровно та же, что будет в DQN.

In [ ]:
import gymnasium as gym

def heuristic_policy(obs):
    x, x_dot, theta, theta_dot = obs
    return int(theta_dot > 0)          # та же эвристика, что на лекции

def collect(policy, n_episodes, seed=0):
    env = gym.make("CartPole-v1")
    X, y = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        while True:
            a = policy(obs)
            X.append(obs); y.append(a)
            obs, r, terminated, truncated, _ = env.step(a)
            if terminated or truncated:
                break
    env.close()
    return torch.as_tensor(np.array(X), dtype=torch.float32), torch.as_tensor(y)

X, y = collect(heuristic_policy, n_episodes=30)
print("датасет:", X.shape, y.shape, "| доля действия 'вправо':", y.float().mean().item())

In [ ]:
policy_net = MLP(in_dim=4, hidden=64, out_dim=2)
optimizer = torch.optim.Adam(policy_net.parameters(), lr=1e-3)

for step in range(1500):
    idx = torch.randint(0, len(X), (64,))              # случайный мини-батч
    logits = policy_net(X[idx])                         # (64, 2) — "оценки" каждого действия
    loss = F.cross_entropy(logits, y[idx])              # softmax + log-loss внутри
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        with torch.no_grad():
            acc = (policy_net(X).argmax(1) == y).float().mean().item()
        print(f"step {step:4d} | loss {loss.item():.3f} | accuracy {acc:.3f}")

In [ ]:
def net_policy(obs):
    with torch.no_grad():                               # применяем сеть — граф не нужен
        logits = policy_net(torch.as_tensor(obs, dtype=torch.float32))
    return int(logits.argmax())

def evaluate(policy, n_episodes=20, seed=100):
    env = gym.make("CartPole-v1")
    lengths = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        lengths.append(total)
    env.close()
    return np.mean(lengths)

print("случайная политика :", evaluate(lambda obs: np.random.randint(2)))
print("эвристика (учитель):", evaluate(heuristic_policy))
print("сеть-клон          :", evaluate(net_policy))

Сеть выучила поведение учителя по 4 числам и держит шест примерно столько же, сколько эвристика. Ограничение очевидно: лучше учителя она не станет. Чтобы стать лучше, нужен сигнал от среды, то есть награда, и способ превратить награду в градиент. Этим и займёмся, начиная с недели 5.

## 6. Распределения: стохастическая политика

В policy gradient (неделя 6) сеть выдаёт не «лучшее действие», а **распределение** над действиями, из которого сэмплируем. Понадобятся `torch.distributions.Categorical` и `log_prob`: градиент REINFORCE — это $\nabla_\theta \log \pi_\theta(a \mid s) \cdot G_t$.

In [ ]:
from torch.distributions import Categorical

fresh_net = MLP(in_dim=4, hidden=64, out_dim=2)   # необученная сеть: вероятности ещё «размазаны»
logits = fresh_net(X[:5])                          # (5, 2)
dist = Categorical(logits=logits)                  # softmax внутри
actions = dist.sample()                    # по одному действию на наблюдение
log_probs = dist.log_prob(actions)         # log π(a|s), через него пойдёт градиент
print("probs:\n", dist.probs.detach().numpy().round(3))
print("actions:", actions.tolist())
print("log_prob:", log_probs.detach().numpy().round(3))
print("entropy:", dist.entropy().detach().numpy().round(3))   # энтропийный бонус в PPO/SAC

# Псевдо-шаг REINFORCE: максимизируем log_prob * return -> минимизируем минус
fake_returns = torch.tensor([1.0, 1.0, -1.0, 0.5, 2.0])
loss = -(log_probs * fake_returns).mean()
loss.backward()
print("градиент есть у первого слоя:", fresh_net.net[0].weight.grad.abs().sum().item() > 0)

## 7. Чек-лист типовых ошибок

1. **Забыли `optimizer.zero_grad()`** — градиенты накапливаются, обучение расходится.
2. **`float64` vs `float32`** — ошибка `expected scalar type Float but found Double`. Лечится `dtype=torch.float32` при создании тензора.
3. **Размерности.** `nn.Linear` ждёт `(batch, features)`. Одно наблюдение из Gymnasium — вектор `(4,)`, при необходимости `unsqueeze(0)`. Лоссы вроде `mse_loss` тихо делают broadcasting между `(batch,)` и `(batch, 1)` и считают ерунду: проверяйте, что формы совпадают.
4. **Не отрезали target** (`detach` / `no_grad`) — градиент течёт через target, Q-обучение неустойчиво.
5. **Считаете статистику с графом.** Логируете `loss` вместо `loss.item()` — граф хранится в памяти, и она течёт.
6. **Сеть в режиме train/eval.** Для MLP без `Dropout`/`BatchNorm` неважно, но привычка вызывать `net.eval()` при оценке и `net.train()` при обучении спасёт позже.
7. **Один seed.** RL шумный: сравнивайте алгоритмы по нескольким seed, а не по одному запуску.

## 8. Упражнения для самопроверки

Необязательные, но полезные: каждое проверяется `assert`.

In [ ]:
# 8.1. Реализуйте функцию, которая по тензору Q-значений (batch, n_actions) и тензору действий (batch,)
#      возвращает Q(s, a) формы (batch,). Используйте gather.
def q_taken(q_values: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

# q = torch.tensor([[1.0, 2.0], [3.0, 4.0]]); a = torch.tensor([1, 0])
# assert torch.allclose(q_taken(q, a), torch.tensor([2.0, 3.0]))

In [ ]:
# 8.2. Посчитайте градиент функции f(x) = sum(x^3) в точке x = [1, 2, 3] через autograd.
#      Ожидаемый ответ: 3 * x^2 = [3, 12, 27].
def grad_of_cube(x: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

# assert torch.allclose(grad_of_cube(torch.tensor([1.0, 2.0, 3.0])), torch.tensor([3.0, 12.0, 27.0]))

In [ ]:
# 8.3. Реализуйте TD-target для батча переходов: target = r + γ * max_a' Q(s', a') * (1 - done).
#      Target не должен требовать градиента (проверяется через requires_grad).
def td_target(rewards, q_next, dones, gamma=0.99):
    # rewards: (batch,), q_next: (batch, n_actions) с requires_grad=True, dones: (batch,) из 0/1
    raise NotImplementedError

# r = torch.tensor([1.0, 0.0]); qn = torch.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True); d = torch.tensor([0.0, 1.0])
# t = td_target(r, qn, d, gamma=0.5)
# assert torch.allclose(t, torch.tensor([2.0, 0.0])) and not t.requires_grad

In [ ]:
# 8.4. Придумайте эвристику лучше лекционной (например, учтите угол шеста и положение тележки),
#      соберите по ней данные и обучите сеть-клон так, чтобы она держала шест в среднем >= 300 шагов.
# assert evaluate(net_policy) >= 300